# Q3 planning: co-author seed and first journal entry

- no real Q3 data are loaded here yet: the topic part still has to arrive
- the small table below is made up so I can check whether I understand the question and the calculation
- this is not meant to be the finished model; it is more like notes for how I think the real analysis should work

**The setup in plain words:**

- one row = one opportunity for an author to enter a journal she has not used before
- `T` = topic fit: how similar is the author's earlier topic history to the journal's earlier topic history?
- `C` = co-author seed: did an earlier co-author already publish in that journal?
- `F` = first entry: did the author publish there for the first observed time in this year?

**Why I call it a seed:**

- example: Ana has never published in journal X
- Ana wrote with Ben earlier, and Ben already published in X
- this gives Ana a possible personal route into X, but it does not prove that Ben caused the later entry

Q3 asks whether entries happen more often when this seed exists, after trying to correct for the fact that co-authors usually work on similar topics.


## Step 0: the assumed picture

`T -> C`, `T -> F`, and `C -> F` is the arrow we want to measure.

- T pushes on both C and F, so T is a confounder: C and F move together through `C <- T -> F` even if `C -> F` does not exist
- the whole plan of Q3 in one sentence: account for T, then see what is left of `C -> F`
- the arrow directions are assumptions, not something the data can decide; "topic brought the collaborators" and "collaborators pulled the topic" produce the same table

Words I keep using: **risk** = share of rows where the event happened (25 of 100 rows enter: risk 0.25). **RR** = one risk divided by the other, 1.0 means no difference. **Risk set** = the rows still able to enter in year t; after a first entry the pair drops out.


## Step 1: define one row before doing any calculation

For author `a`, year `t` and journal `X`:

- the author must not have an earlier X publication in our study window
- `C = 1` needs two earlier facts at once: the collaboration happened before year t AND that co-author's X publication is before year t; otherwise `C = 0`
- `F = 1` if the author makes a first observed X publication during year t, otherwise `F = 0`

Timing rule: `history uses years < t; outcome uses year t`. We stay at year level on purpose, because OpenAlex fills missing dates with 1 January and any finer order would be partly invented.

`F = 0` does not mean rejected. It only means no successful X publication was observed that year.


## Step 2: make a tiny table by hand

- invented rows, small enough to check by eye
- the column `T` is only a placeholder for the later topic data


In [1]:
import pandas as pd

# made-up examples, not project results
# the T column is guesswork, I have no idea yet what the real spread looks like
toy = pd.DataFrame([
    ["Ana",   2020, "J1", 1, 0.80, 1],
    ["Ana",   2020, "J2", 0, 0.55, 0],
    ["Bo",    2020, "J1", 0, 0.72, 0],
    ["Bo",    2020, "J3", 1, 0.20, 0],
    ["Chen",  2021, "J2", 1, 0.65, 1],
    ["Chen",  2021, "J3", 0, 0.60, 0],
    ["Deepa", 2021, "J1", 1, 0.75, 0],
    ["Deepa", 2021, "J3", 0, 0.30, 1],
], columns=["author", "year", "journal", "C", "T", "F"])

toy


,author,year,journal,C,T,F
0,Ana,2020,J1,1,0.80,1
1,Ana,2020,J2,0,0.55,0
2,Bo,2020,J1,0,0.72,0
3,Bo,2020,J3,1,0.20,0
4,Chen,2021,J2,1,0.65,1
5,Chen,2021,J3,0,0.60,0
6,Deepa,2021,J1,1,0.75,0
7,Deepa,2021,J3,0,0.30,1


Things I notice:

- one author creates several rows, so rows are not independent
- a seed is not a guarantee: Bo-J3 has `C = 1` but `F = 0`
- entry without a seed happens: Deepa-J3 has `C = 0` but `F = 1`
- after a first entry, that author-journal pair should disappear from later risk sets
- still unsure: if an author enters two unfamiliar journals in the same year, both rows get `F = 1` and I have not decided whether they stay independent rows or need to be treated as one choice


## Step 3: the naive comparison

Split rows into `C = 1` and `C = 0`, compute `risk = entries / rows` per group, divide.


In [2]:
# raw counting only: no correction for topic or author differences yet
raw = toy.groupby("C")["F"].agg(rows="size", entries="sum", risk="mean")

raw_rr = raw.loc[1, "risk"] / raw.loc[0, "risk"]

print(raw)
print(f"raw RR = {raw_rr:.2f}")


   rows  entries  risk
C                     
0     4        1  0.25
1     4        2  0.50
raw RR = 2.00


So: 0.50 with seed, 0.25 without, raw RR = 2.00.

- 0.50 against 0.25 is counting and nothing more. The moment I write "so co-authors double the rate" I have added a cause that the counting does not support
- only arithmetic on eight invented rows, not evidence
- even on real rows this ratio would mix the seed with topic fit, productivity, network size, journal and year
- I still want to show the naive number on real data as the starting point; watching it move under adjustment is more convincing than only claiming it was contaminated


## Step 4: why topic can fake part of the pattern

- authors collaborate with people who work on similar topics
- a shared topic makes the same journals plausible for both people, so seeds pile up exactly where entry was likely anyway
- comparing the two piles therefore compares seed AND topic at the same time, in unknown proportions; that is the backdoor path `C <- T -> F` from Step 0
- the standard example: people who carry a lighter get lung cancer more often, but smoking causes both the lighter and the cancer. Topic is the smoking here, the co-author is the lighter
- a clean time lag alone does not fix this: Ben being in X first does not rule out the shared topic or group that connected Ana and Ben in the first place

This is why I would call the final number a **topic-adjusted network association**, unless the stronger causal assumptions can really be defended.


## Step 5: a possible topic-fit measure

- Ana's earlier papers: 70% machine learning, 30% robotics
- journal X's earlier papers: 60% machine learning, 40% robotics
- `topic fit = 0.70 * 0.60 + 0.30 * 0.40 = 0.54`, a similarity score, not a 54% probability of anything
- the measure should come from titles, abstracts and keywords, not from the journal's own subject label; OpenAlex partly derives topic from the venue, and that would be circular


In [3]:
# same hand calculation in code, just to check it
ana = {"machine_learning": 0.70, "robotics": 0.30}
journal_x = {"machine_learning": 0.60, "robotics": 0.40}

topic_fit = sum(ana[topic] * journal_x[topic] for topic in ana)
print("topic-fit example =", round(topic_fit, 2))


topic-fit example = 0.54


## Step 6: adjust for T by asking twice

The later model is roughly `F ~ C + T + earlier covariates`. The coefficient is not the answer. Instead:

1. predict every row with `C = 1`, each row keeps its real `T`
2. predict the same rows with `C = 0`, same `T` values again
3. average each side, divide

Both imagined worlds contain the same authors and the same topic mix; they differ only in C. The cell below does the same thing by hand, without a model.


In [4]:
# invented probabilities chosen only to make the adjustment visible
share_good_fit = 0.40

# entry risks within the two topic-fit groups
risk_good_fit_C1, risk_good_fit_C0 = 0.80, 0.50
risk_poor_fit_C1, risk_poor_fit_C0 = 0.40, 0.10

def adjusted_rr(good_C1, good_C0, poor_C1, poor_C0, share_good=share_good_fit):
    """the SAME topic mixture on both sides"""
    risk_C1 = share_good * good_C1 + (1 - share_good) * poor_C1
    risk_C0 = share_good * good_C0 + (1 - share_good) * poor_C0
    return risk_C1, risk_C0, risk_C1 / risk_C0

def naive_rr(good_C1, good_C0, poor_C1, poor_C0):
    """each side keeps its own mixture, seeds pile up where topics fit"""
    risk_C1 = 0.80 * good_C1 + 0.20 * poor_C1
    risk_C0 = 0.20 * good_C0 + 0.80 * poor_C0
    return risk_C1, risk_C0, risk_C1 / risk_C0

a1, a0, a_rr = adjusted_rr(risk_good_fit_C1, risk_good_fit_C0, risk_poor_fit_C1, risk_poor_fit_C0)
n1, n0, n_rr = naive_rr(risk_good_fit_C1, risk_good_fit_C0, risk_poor_fit_C1, risk_poor_fit_C0)

print(f"adjusted risks: {a1:.2f} vs {a0:.2f}  ->  RR {a_rr:.2f}")
print(f"naive risks:    {n1:.2f} vs {n0:.2f}  ->  RR {n_rr:.2f}")

adjusted risks: 0.56 vs 0.26  ->  RR 2.15
naive risks:    0.72 vs 0.18  ->  RR 4.00


What comes out:

- adjusted 0.56 vs 0.26, RR 2.15; absolute difference 0.30, always shown next to the ratio
- naive RR 4.00 from the same four risks, only because the C=1 side got the topic-rich 80/20 mixture and the C=0 side the 20/80 one
- the only difference between the two lines is which weights each side gets
- 2.15 does not prove the co-author caused anything; adjustment only removes what was measured
- not automatically a direct effect, not automatically a lower bound


## Step 7: the three textbook words for Step 6, in normal language

**Stratify.** A stratum is just a box of rows that share the same topic situation: one box "topic fits", one box "topic does not fit". Stratifying means sorting rows into the boxes and only comparing inside a box, so topic cannot explain a gap there. Smoking version: compare lighter-carriers vs non-carriers among smokers only, then again among non-smokers only; inside each box, smoking cannot be the reason.

**Standardize.** Recombine the box results using ONE mixing recipe for both sides, the real field mix (40/60 here). The naive comparison quietly gives each side its own recipe (80/20 vs 20/80). That substitution is the whole 4.0-vs-2.15 gap. As a formula, with the four box risks plugged in:

`P(F=1 | do(C=c)) = sum over t of P(F=1 | C=c, T=t) * P(T=t)`

c=1: `0.80*0.4 + 0.40*0.6 = 0.56`. c=0: `0.26`. Note it uses `P(T)`, not `P(T | C)`; the naive version uses `P(T | C)` without saying so.

**do().** The thought experiment "what if every row got the seed / lost the seed". It applies to C, never to T; we are not asking what happens if we change someone's topic. On the diagram, do(C) would cut the arrow `T -> C`; in the calculation, overriding the C column while T stays real is the stand-in for that cut. Important: do() names the ideal target, not something we achieved. Our number equals it only if the assumptions hold (T measured well enough, no hidden common causes like institution or seniority, enough overlap between seeded and unseeded rows, and T measured before C begins). That is exactly why the reporting name stays topic-adjusted network association.

Two small notes:

- the backdoor path `C <- T -> F` closes at the moment T is in the model; the ask-twice averaging then does the second job, giving both worlds the same topic mix
- the good/poor boxes are a teaching device; real T is continuous, and `F ~ C + T` is the smooth version of the boxes, so we never actually bin T into two levels


## Step 8: a very small calibration check

Like calibrating a scale: put a known 1 kg weight on it, it must show 1 kg; put nothing on it, it must show 0. Here I build a world where the seed changes nothing (0.50 both seed conditions where topic fits, 0.10 where not) and check what both methods report.


In [5]:
# the point of the check is that these are the SAME two functions as Step 6,
# only the risk table changes. Writing the null result out by hand would prove nothing.
_, _, null_adjusted = adjusted_rr(0.50, 0.50, 0.10, 0.10)   # seed changes nothing inside either box
_, _, null_naive    = naive_rr(0.50, 0.50, 0.10, 0.10)

print(f"planted-null adjusted RR = {null_adjusted:.2f}")
print(f"planted-null naive RR    = {null_naive:.2f}")

planted-null adjusted RR = 1.00
planted-null naive RR    = 2.33


Result:

- adjusted returns exactly 1.00 when the planted truth is "no effect"
- naive still reports 2.33 in the same world, produced entirely by the uneven topic spread. A method that reports an effect here is useless for the real question
- this checks the calculator on a world with a known answer; it does not show the diagram is right, that T is measured well, or that hidden variables are absent
- the fitted-model version of this test (sampled rows, bootstrap intervals) is not written yet, it needs the real pipeline first


**One timing choice that is not solved yet:**

- rolling fit: topic history up to end of `t-1`; current, but a co-author may already have influenced that history
- baseline fit: frozen before the first seed appears; cleaner timing, but possibly stale for late entries
- what is at stake: for T to be a clean confounder it should be measured before C begins; otherwise part of the diagram reads `C -> T -> F` and adjusting removes part of the influence we want to study
- plan: run both side by side; if they agree the choice does not matter, if they differ the gap is itself a result


## Step 9: what the real analysis would do later

1. use publication years only and freeze history at the end of `t-1`
2. build the earlier co-author network
3. create one row for every active author-year and unfamiliar journal
4. mark `T`, `C`, `F` and earlier author history
5. print the raw risks first as a data check
6. fit a logistic model with topic fit, earlier papers, earlier co-authors, career age, journal breadth, journal and year
7. predict the same rows once with `C = 1` and once with `C = 0`
8. average both sets of predicted risks and divide them to get the adjusted RR
9. resample whole authors for the interval, because each author creates many related rows

I am not fitting that model here because the needed data are not available. Adding a full model now would make the notebook look more finished than it really is.


**Checks planned for later:**

- no paper from year t is allowed to create earlier exposure in the same year
- after first entry, the author-journal pair is removed
- connected and unconnected rows need some overlap in topic fit and author history
- main version: separate journal effects and year effects; sensitivity: journal-by-year, but only if the cells contain enough entries and both exposure values
- repeat the topic adjustment with rolling fit and baseline fit
- separate entries made with the experienced co-author on the paper from entries made without that person (riding along check)
- repeat the calibration with the final pipeline: plant one positive effect and one true null; both directions must be recovered (checks the calculator, not the diagram)


## Placeholder for the real files

Just a reminder of the columns I think we will need. No file is loaded here yet.


In [6]:
expected_paper_columns = {
    "paper_id", "journal_id", "publisher_id",
    "publication_year", "subject_label"
}
expected_authorship_columns = {"paper_id", "author_id"}

print("papers:", sorted(expected_paper_columns))
print("authorships:", sorted(expected_authorship_columns))
print("status: waiting for the real topic / authorship input")


papers: ['journal_id', 'paper_id', 'publication_year', 'publisher_id', 'subject_label']
authorships: ['author_id', 'paper_id']
status: waiting for the real topic / authorship input


## Questions I should be able to answer out loud

1. why is a row an opportunity and not a paper? (what is the denominator, and does the publication table even contain one for Q3?)
2. in "the RR is 2.0, so co-authors double the entry rate", which part is counting and which is unearned?
3. which variable gets stratified, which gets standardized over, which one does do() apply to?
4. why does the naive comparison secretly use `P(T | C)` instead of `P(T)`, and why does that explain the whole 4.0 vs 2.15 gap?
5. why does recovering a planted answer prove nothing about the diagram?
6. why two timing variants of T, and what does it mean if they disagree?


## Current takeaway

- the eventual headline is an adjusted risk ratio, with the two risks shown beside it
- the data observe successful publication, not submission or rejection
- safest claim: an earlier co-author seed marks a higher or lower probability of first observed journal entry after the stated adjustments
- do(C) names the target we would love to have; the number we can defend is the topic-adjusted network association
- this notebook gets expanded when the topic data arrive; the made-up values above must never be reported as project findings
